# DECONGESTILAGUNA - Garbage Detection YOLOv8 Training
**Auto-label with Claude Haiku + Fine-tune YOLOv8 on Google Colab**

This notebook:
1. Uploads your CCTV frames
2. Auto-labels garbage/trash using Claude Haiku vision
3. Fine-tunes YOLOv8s on your labeled data
4. Exports the trained model (.pt) for deployment to your Pi

---

## Step 1: Setup Environment

In [ ]:
# Install required packages
!pip install ultralytics anthropic pillow opencv-python-headless -q

import os, json, base64, cv2, shutil, glob, random
import numpy as np
from pathlib import Path
from google.colab import files, userdata
from IPython.display import display, Image as IPImage, clear_output

print('Environment ready!')

## Step 2: Configure API Key & Classes

Add your Anthropic API key in Colab: **Secrets (key icon on left sidebar)** → Add `ANTHROPIC_API_KEY`

In [ ]:
# Get API key from Colab secrets
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except:
    ANTHROPIC_API_KEY = input('Enter your Anthropic API key: ')

# Class definitions - same as your cctv_ai model
CLASS_NAMES = [
    'BASKET', 'BOTTLE', 'BOX', 'BUCKET', 'CAN', 'CANAL', 'CARDBOARD',
    'CHAIR', 'CONTAINER', 'CRATE', 'CUP', 'FALLEN_TREE', 'GARBAGE',
    'GROCERY_BAG', 'LEAVES', 'OPEN_CANAL', 'PAPER', 'PLASTIC',
    'PLASTIC_BOTTLE', 'PLASTIC_CONTAINER', 'PLASTIC_BAG', 'POT', 'ROCK',
    'SACK', 'TISSUE', 'TRASH', 'TRASH_CAN', 'VENDOR'
]

CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

print(f'{NUM_CLASSES} classes configured')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i}: {name}')

## Step 3: Upload CCTV Frames

Upload your collected frames (JPG/PNG). Aim for **200-500 images** for good results.

**Option A:** Upload from your computer  
**Option B:** Upload a ZIP file of frames  
**Option C:** Pull frames from your Pi directly

In [ ]:
# Create directories
DATASET_DIR = Path('/content/garbage_dataset')
RAW_DIR = DATASET_DIR / 'raw_frames'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# === OPTION A: Upload individual images ===
print('Upload your CCTV frames (JPG/PNG):')
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(RAW_DIR / fname, 'wb') as f:
        f.write(data)
print(f'Uploaded {len(uploaded)} files')

In [ ]:
# === OPTION B: Upload a ZIP file instead ===
# Uncomment below if you prefer uploading a ZIP

# print('Upload a ZIP file of frames:')
# uploaded = files.upload()
# for fname in uploaded:
#     !unzip -o "{fname}" -d {RAW_DIR}
# print('Extracted!')

In [ ]:
# === OPTION C: Pull frames from Pi via SSH ===
# Uncomment and fill in your Pi details

# !pip install paramiko -q
# import paramiko
# ssh = paramiko.SSHClient()
# ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
# ssh.connect('YOUR_PI_PUBLIC_URL', username='admin', password='admin123')
# sftp = ssh.open_sftp()
# remote_dir = '/home/admin/illegal-parking/static/violations/'
# for f in sftp.listdir(remote_dir):
#     if f.endswith('.jpg'):
#         sftp.get(remote_dir + f, str(RAW_DIR / f))
# sftp.close()
# ssh.close()

In [ ]:
# Check how many frames we have
all_frames = sorted(glob.glob(str(RAW_DIR / '*.jpg')) + glob.glob(str(RAW_DIR / '*.png')))
print(f'Total frames available: {len(all_frames)}')

# Show a few samples
for img_path in all_frames[:3]:
    display(IPImage(filename=img_path, width=400))

## Step 4: Auto-Label with Claude Haiku

Claude Haiku will analyze each frame and generate YOLO-format labels automatically.

In [ ]:
import anthropic
import re

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def auto_label_frame(image_path):
    """
    Use Claude Haiku to detect garbage objects and return YOLO annotations.
    Returns list of [class_id, x_center, y_center, width, height] (normalized 0-1)
    """
    img = cv2.imread(image_path)
    if img is None:
        return []
    h, w = img.shape[:2]

    # Resize for API (keep aspect ratio)
    send_w = 800
    send_h = int(800 * h / w)
    small = cv2.resize(img, (send_w, send_h))
    _, buf = cv2.imencode('.jpg', small, [cv2.IMWRITE_JPEG_QUALITY, 85])
    img_b64 = base64.b64encode(buf).decode('utf-8')

    class_list = ', '.join(CLASS_NAMES)

    prompt = f"""You are a precise object detection labeling system for CCTV garbage/waste detection.
The image is {send_w}x{send_h} pixels from a street camera in the Philippines.

Detect ALL objects that belong to these classes:
{class_list}

Respond ONLY with valid JSON (no markdown):
{{
  "objects": [
    {{
      "class": "exact class name from the list above",
      "bbox": [x1, y1, x2, y2]
    }}
  ]
}}

CRITICAL RULES:
- bbox values are PIXEL coordinates in this {send_w}x{send_h} image
- x1,y1 = top-left corner, x2,y2 = bottom-right corner of the object
- Draw TIGHT boxes around each object - no extra padding
- Only use class names from the list above (exact spelling)
- If no objects found, return empty objects list
- Be thorough: detect every piece of garbage, trash, container, etc."""

    try:
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=1000,
            messages=[{
                'role': 'user',
                'content': [
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': img_b64}},
                    {'type': 'text', 'text': prompt}
                ]
            }]
        )

        raw = response.content[0].text.strip()
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        result = json.loads(match.group() if match else raw)

        annotations = []
        for obj in result.get('objects', []):
            cls_name = obj['class'].upper()
            if cls_name not in CLASS_TO_ID:
                continue
            cls_id = CLASS_TO_ID[cls_name]
            x1, y1, x2, y2 = obj['bbox']

            # Convert pixel coords to YOLO format (normalized center + size)
            x_center = ((x1 + x2) / 2) / send_w
            y_center = ((y1 + y2) / 2) / send_h
            bw = abs(x2 - x1) / send_w
            bh = abs(y2 - y1) / send_h

            # Clamp to valid range
            x_center = max(0, min(1, x_center))
            y_center = max(0, min(1, y_center))
            bw = max(0.01, min(1, bw))
            bh = max(0.01, min(1, bh))

            annotations.append([cls_id, x_center, y_center, bw, bh])

        return annotations

    except Exception as e:
        print(f'  Error: {e}')
        return []

print('Auto-labeling function ready!')

In [ ]:
# Run auto-labeling on all frames
LABELS_DIR = DATASET_DIR / 'labels_raw'
LABELS_DIR.mkdir(parents=True, exist_ok=True)

labeled_count = 0
empty_count = 0

for i, img_path in enumerate(all_frames):
    fname = Path(img_path).stem
    label_path = LABELS_DIR / f'{fname}.txt'

    # Skip if already labeled
    if label_path.exists():
        labeled_count += 1
        continue

    print(f'[{i+1}/{len(all_frames)}] Labeling {Path(img_path).name}...', end=' ')
    annotations = auto_label_frame(img_path)

    if annotations:
        with open(label_path, 'w') as f:
            for ann in annotations:
                f.write(f'{ann[0]} {ann[1]:.6f} {ann[2]:.6f} {ann[3]:.6f} {ann[4]:.6f}\n')
        labeled_count += 1
        print(f'{len(annotations)} objects found')
    else:
        # Write empty label file (negative sample)
        label_path.touch()
        empty_count += 1
        print('no objects')

print(f'\nDone! {labeled_count} labeled, {empty_count} empty (negative samples)')

## Step 5: Review Labels (Visual Check)

Visualize the auto-generated labels to verify quality before training.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

COLORS = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))

def visualize_labels(img_path, label_path, ax=None):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img)
    ax.set_title(Path(img_path).name, fontsize=10)
    ax.axis('off')

    if not os.path.exists(label_path):
        return
    with open(label_path) as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id = int(parts[0])
        xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        x1 = (xc - bw/2) * w
        y1 = (yc - bh/2) * h
        box_w = bw * w
        box_h = bh * h
        color = COLORS[cls_id % len(COLORS)]
        rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2,
                                  edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        label = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'cls{cls_id}'
        ax.text(x1, y1-5, label, color='white', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

# Show random samples
labeled_frames = [f for f in all_frames if (LABELS_DIR / (Path(f).stem + '.txt')).exists()
                  and (LABELS_DIR / (Path(f).stem + '.txt')).stat().st_size > 0]

samples = random.sample(labeled_frames, min(6, len(labeled_frames)))
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, samples):
    label_path = str(LABELS_DIR / (Path(img_path).stem + '.txt'))
    visualize_labels(img_path, label_path, ax)
for ax in axes.flat[len(samples):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Showing {len(samples)} random labeled samples. Review the boxes!')
print('If labels look wrong, you can manually fix the .txt files in labels_raw/')

## Step 6: Prepare YOLO Dataset

Split into train/val sets and create the `data.yaml` config.

In [ ]:
# Create YOLO dataset structure
YOLO_DIR = Path('/content/yolo_dataset')
for split in ['train', 'val']:
    (YOLO_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

# Get all labeled frames
all_labeled = []
for img_path in all_frames:
    label_path = LABELS_DIR / (Path(img_path).stem + '.txt')
    if label_path.exists():
        all_labeled.append((img_path, str(label_path)))

# Shuffle and split 85/15
random.shuffle(all_labeled)
split_idx = int(len(all_labeled) * 0.85)
train_set = all_labeled[:split_idx]
val_set = all_labeled[split_idx:]

def copy_set(dataset, split_name):
    for img_path, lbl_path in dataset:
        fname = Path(img_path).name
        lbl_name = Path(img_path).stem + '.txt'
        shutil.copy2(img_path, YOLO_DIR / split_name / 'images' / fname)
        shutil.copy2(lbl_path, YOLO_DIR / split_name / 'labels' / lbl_name)

copy_set(train_set, 'train')
copy_set(val_set, 'val')

print(f'Dataset split: {len(train_set)} train, {len(val_set)} val')

# Create data.yaml
data_yaml = {
    'path': str(YOLO_DIR),
    'train': 'train/images',
    'val': 'val/images',
    'names': {i: name for i, name in enumerate(CLASS_NAMES)}
}

import yaml
with open(YOLO_DIR / 'data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f'data.yaml created at {YOLO_DIR / "data.yaml"}')
print('\nClass distribution:')

# Count class distribution
class_counts = {i: 0 for i in range(NUM_CLASSES)}
for _, lbl_path in all_labeled:
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1
for cls_id, count in sorted(class_counts.items()):
    if count > 0:
        print(f'  {CLASS_NAMES[cls_id]}: {count}')

## Step 7: Fine-Tune YOLOv8

Fine-tune from your existing `cctv_ai.pt` model (or start from `yolov8s.pt`).

In [ ]:
# Upload your existing cctv_ai.pt model for fine-tuning (recommended)
# This preserves what the model already learned and improves it

print('Upload your existing cctv_ai.pt model for fine-tuning.')
print('(If you skip this, training will start from yolov8s.pt instead)')
print()

BASE_MODEL = 'yolov8s.pt'  # default fallback

try:
    uploaded_model = files.upload()
    for fname in uploaded_model:
        if fname.endswith('.pt'):
            with open(f'/content/{fname}', 'wb') as f:
                f.write(uploaded_model[fname])
            BASE_MODEL = f'/content/{fname}'
            print(f'Using uploaded model: {fname}')
            break
except:
    print(f'No model uploaded, using {BASE_MODEL}')

print(f'\nBase model: {BASE_MODEL}')

In [ ]:
from ultralytics import YOLO

# Load base model
model = YOLO(BASE_MODEL)

# Train!
results = model.train(
    data=str(YOLO_DIR / 'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,           # Early stopping if no improvement for 20 epochs
    device=0,              # GPU
    workers=2,
    project='/content/runs',
    name='garbage_detector',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    degrees=10,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    verbose=True
)

print('\nTraining complete!')

## Step 8: Evaluate Results

In [ ]:
# Show training results
results_dir = Path('/content/runs/garbage_detector')

# Display training curves
if (results_dir / 'results.png').exists():
    display(IPImage(filename=str(results_dir / 'results.png'), width=800))

# Display confusion matrix
if (results_dir / 'confusion_matrix.png').exists():
    display(IPImage(filename=str(results_dir / 'confusion_matrix.png'), width=600))

# Display validation predictions
if (results_dir / 'val_batch0_pred.jpg').exists():
    display(IPImage(filename=str(results_dir / 'val_batch0_pred.jpg'), width=800))

print('\nBest model saved at:', results_dir / 'weights' / 'best.pt')

In [ ]:
# Run validation metrics
best_model = YOLO(str(results_dir / 'weights' / 'best.pt'))
metrics = best_model.val(data=str(YOLO_DIR / 'data.yaml'))

print(f'\nmAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

## Step 9: Test on Sample Images

In [ ]:
# Test the trained model on validation images
val_images = sorted(glob.glob(str(YOLO_DIR / 'val' / 'images' / '*')))
test_samples = random.sample(val_images, min(4, len(val_images)))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, img_path in zip(axes.flat, test_samples):
    results = best_model(img_path, conf=0.2)[0]
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        label = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'cls{cls_id}'
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 165, 255), 2)
        cv2.putText(img, f'{label} {conf:.0%}', (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    ax.imshow(img)
    ax.set_title(Path(img_path).name)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 10: Export & Download

Download the trained model to deploy on your Raspberry Pi.

In [ ]:
# Export to ONNX as well (for Hailo conversion later)
best_model.export(format='onnx', imgsz=640, simplify=True)
print('ONNX export complete!')

In [ ]:
# Download the trained models
best_pt = str(results_dir / 'weights' / 'best.pt')
best_onnx = best_pt.replace('.pt', '.onnx')

# Copy to easy download location
shutil.copy2(best_pt, '/content/cctv_ai_v2.pt')
if os.path.exists(best_onnx):
    shutil.copy2(best_onnx, '/content/cctv_ai_v2.onnx')

print('Downloading trained model...')
files.download('/content/cctv_ai_v2.pt')

print('\n=== DEPLOYMENT INSTRUCTIONS ===')
print('1. Copy cctv_ai_v2.pt to your Pi: ~/illegal-parking/models/')
print('2. Rename: mv models/cctv_ai.pt models/cctv_ai_v1_backup.pt')
print('3. Rename: mv models/cctv_ai_v2.pt models/cctv_ai.pt')
print('4. Restart: sudo systemctl restart parking-detect')
print()
print('For Hailo (.hef) conversion, use the Hailo Model Zoo DFC tool:')
print('  hailo optimize cctv_ai_v2.onnx --hw-arch hailo8l')
print('  hailo compile cctv_ai_v2.har --hw-arch hailo8l')

---
## Optional: Save Dataset for Future Training

Download the labeled dataset so you can retrain later with more data.

In [ ]:
# Zip and download the full labeled dataset
!cd /content && zip -r garbage_dataset.zip yolo_dataset/ -q
files.download('/content/garbage_dataset.zip')
print('Dataset saved! You can re-upload this next time to add more data.')